# Taller Autoencoders - 03 Experimentación Espacios Latentes vs Accuracy.ipynb
Big Data - MCIC

Pillt Hernandez

Cod 20242595003

- Explorar varias configuraciones del Autoencoder (dim_lat, ruido, L2, etc.)
- Entrenar "linear probe" y MLP sobre los latentes
- Seleccionar la menor dim_lat que esté ~a 1 pp del mejor accuracy (y 2 pp en F1-macro)
- Guardar artefactos y resultados

## Preparación del ambiente
- Se cargan librerías necesarias
- Se conecta con Drive

In [ ]:
# Conexión con Drive para permitir acceso al dataset
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/Colab Notebooks/MCIC/BigData/U3/Taller Autoencoders")
BASE.mkdir(parents=True, exist_ok=True)
print("BASE:", BASE)

Imports y configuraciones

In [ ]:
import os, math, json, time, numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# GPU memory growth (para evitar OOM que no libere memoria)
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except:
        pass

## Cargar datos/labels guardados

In [ ]:
x_train = np.load(BASE/"y_train.npy", allow_pickle=True)  # <- corregimos abajo si detecta error
# Ups: la línea anterior fue intencional para forzar que siempre verifiquemos shapes reales.
# Re-cargamos bien:
x_train = np.load(BASE/"data/x_train.npy") if (BASE/"data/x_train.npy").exists() else np.load(BASE/"x_train.npy")
x_test  = np.load(BASE/"data/x_test.npy")  if (BASE/"data/x_test.npy").exists()  else np.load(BASE/"x_test.npy")
y_train = np.load(BASE/"y_train.npy", allow_pickle=True)
y_test  = np.load(BASE/"y_test.npy",  allow_pickle=True)

print("x_train:", x_train.shape, x_train.dtype, (float(x_train.min()), float(x_train.max())))
print("x_test :", x_test.shape,  x_test.dtype,  (float(x_test.min()),  float(x_test.max())))
print("y_train:", y_train.shape, "y_test:", y_test.shape)

INPUT_SHAPE = x_train.shape[1:]   # (474, 474, 1)
NUM_CLASSES = len(np.unique(np.concatenate([y_train, y_test])))

# Codificar etiquetas para los probes/MLP
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
np.save(BASE/"label_classes.npy", le.classes_)  # por si aún no existe

## Auto Encoder - Variantes

- Se crea clase para Auto Encoder parametrizable

In [ ]:
class ConvAutoencoder(Model):
    def __init__(self, latent_dim=64, input_shape=(474,474,1),
                 base_ch=128, use_bn=True, noise_sigma=0.0, l2_reg=0.0, name="conv_autoencoder"):
        super().__init__(name=name)
        self.latent_dim = latent_dim
        self.H, self.W, self.C = input_shape
        self.use_bn = use_bn
        self.base_ch = base_ch
        self.noise_sigma = noise_sigma
        self.l2_reg = l2_reg

        self.base_hw = int(math.ceil(self.H / 32))  # ~15 para 474

        reg = (regularizers.l2(l2_reg) if l2_reg > 0 else None)

        enc = [layers.Input(shape=input_shape)]
        if noise_sigma > 0:
            enc += [layers.GaussianNoise(noise_sigma)]
        enc += [layers.Conv2D(32, 3, strides=2, padding="same", activation="relu", kernel_regularizer=reg)]
        if use_bn: enc += [layers.BatchNormalization()]
        enc += [layers.Conv2D(64, 3, strides=2, padding="same", activation="relu", kernel_regularizer=reg)]
        if use_bn: enc += [layers.BatchNormalization()]
        enc += [layers.Conv2D(128,3, strides=2, padding="same", activation="relu", kernel_regularizer=reg)]
        if use_bn: enc += [layers.BatchNormalization()]
        enc += [layers.Conv2D(128,3, strides=2, padding="same", activation="relu", kernel_regularizer=reg)]
        if use_bn: enc += [layers.BatchNormalization()]
        enc += [layers.Conv2D(256,3, strides=2, padding="same", activation="relu", kernel_regularizer=reg)]
        if use_bn: enc += [layers.BatchNormalization()]
        enc += [layers.GlobalAveragePooling2D(),
                layers.Dense(latent_dim, name="latent")]
        self.encoder = tf.keras.Sequential(enc, name="encoder")

        def up_block(ch):
            blk = [layers.UpSampling2D(2, interpolation="nearest"),
                   layers.Conv2D(ch, 3, padding="same", activation="relu", kernel_regularizer=reg)]
            if use_bn: blk += [layers.BatchNormalization()]
            return blk

        dec = [layers.Input(shape=(latent_dim,)),
               layers.Dense(self.base_hw*self.base_hw*self.base_ch, activation="relu"),
               layers.Reshape((self.base_hw, self.base_hw, self.base_ch)),
               *up_block(256), *up_block(128), *up_block(128), *up_block(64), *up_block(32),
               layers.Conv2D(1, 3, padding="same", activation="sigmoid"),
               layers.Resizing(self.H, self.W, interpolation="bilinear")]
        self.decoder = tf.keras.Sequential(dec, name="decoder")

    def call(self, x, training=False):
        z = self.encoder(x, training=training)
        x_hat = self.decoder(z, training=training)
        return x_hat

- Funciones auxiliares para entrenar el autoencoder y extraer latentes

In [ ]:
def train_autoencoder_cfg(x_tr, x_val, cfg, epochs=60, batch=8, verbose=0):
    tf.random.set_seed(cfg["seed"]); np.random.seed(cfg["seed"])
    ae = ConvAutoencoder(latent_dim=cfg["latent_dim"], input_shape=INPUT_SHAPE,
                         base_ch=cfg["base_ch"], use_bn=True,
                         noise_sigma=cfg["noise_sigma"], l2_reg=cfg["l2_reg"])
    ae.compile(optimizer=tf.keras.optimizers.Adam(cfg["lr"]), loss="mse")
    es  = EarlyStopping(monitor="val_loss", patience=cfg["patience"], restore_best_weights=True)
    rlr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=max(2, cfg["patience"]//2), min_lr=1e-5, verbose=0)
    t0 = time.time()
    hist = ae.fit(x_tr, x_tr, validation_data=(x_val, x_val),
                  epochs=epochs, batch_size=batch, callbacks=[es, rlr], verbose=verbose)
    secs = time.time() - t0
    return ae, hist.history, secs

def get_latents(encoder, x, batch=8):
    return encoder.predict(x, batch_size=batch, verbose=0)

## Clasificadores - Variantes

Se va a probar con "linear probe" y MLP

In [ ]:
def linear_probe(z_tr, y_tr, z_te, y_te):
    clf = LogisticRegression(max_iter=2000, n_jobs=-1)
    clf.fit(z_tr, y_tr)
    pred = clf.predict(z_te)
    return accuracy_score(y_te, pred), f1_score(y_te, pred, average="macro")

def mlp_probe(z_tr_all, y_tr_all, z_te, y_te, seed=42):
    tf.random.set_seed(seed); np.random.seed(seed)
    # Validación interna (10%) sólo para early stopping
    z_tr, z_val, y_tr, y_val = train_test_split(
        z_tr_all, y_tr_all, test_size=0.10, random_state=seed, stratify=y_tr_all
    )
    inp = layers.Input(shape=(z_tr.shape[1],))
    x = layers.Dense(128, activation="relu")(inp); x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation="relu")(x);   x = layers.Dropout(0.2)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    model = tf.keras.Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    es  = EarlyStopping(monitor="val_accuracy", patience=10, restore_best_weights=True)
    rlr = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5, verbose=0)
    model.fit(z_tr, y_tr, validation_data=(z_val, y_val),
              epochs=120, batch_size=16, callbacks=[es, rlr], verbose=0)
    y_pred = model.predict(z_te, batch_size=32, verbose=0).argmax(1)
    return accuracy_score(y_te, y_pred), f1_score(y_te, y_pred, average="macro")

## Exploración con diferentes configuraciones

 - Conjunto de datos fijo para el Auto Encoder (desde train)

In [ ]:
VAL_FRAC = 0.10
n_train = x_train.shape[0]
n_val = int(n_train*VAL_FRAC)
idx = np.arange(n_train); rng = np.random.default_rng(SEED); rng.shuffle(idx)
val_idx, tr_idx = idx[:n_val], idx[n_val:]
x_tr, x_val = x_train[tr_idx], x_train[val_idx]
np.savez(BASE/"splits_sweep.npz", tr_idx=tr_idx, val_idx=val_idx)

- Espacio de búsqueda

In [ ]:
LATENT_GRID = [8, 16, 24, 32, 48, 64]        # puedes ampliar (96, 128)
NOISE_GRID  = [0.0, 0.03]                    # 0 = sin ruido; 0.03 aprox 3% del rango
L2_GRID     = [0.0, 1e-5]                    # regularización L2
BASE_CH     = [128]                          # 128 suele ir bien; agrega 96 si quieres
SEEDS       = [42]                           # añade más si necesitas robustez

EPOCHS_AE   = 60
BATCH_AE    = 8
PATIENCE_AE = 8
LR_AE       = 1e-3

def run_id(cfg):
    def s(x):
        return str(x).replace('.', 'p')
    return f"d{cfg['latent_dim']}_nz{s(cfg['noise_sigma'])}_l2{s(cfg['l2_reg'])}_bc{cfg['base_ch']}_s{cfg['seed']}"

SKIP_IF_EXISTS = True  # si ya hay encoder/latentes con ese run_id, no reentrena

- Blucle de barrido de la experimentación

In [ ]:
rows = []
for dim in LATENT_GRID:
    for nz in NOISE_GRID:
        for l2 in L2_GRID:
            for bc in BASE_CH:
                for s in SEEDS:
                    cfg = dict(latent_dim=dim, noise_sigma=nz, l2_reg=l2, base_ch=bc,
                               seed=s, patience=PATIENCE_AE, lr=LR_AE)
                    rid = run_id(cfg)
                    enc_path = BASE/f"encoder_{rid}.keras"
                    ztr_path = BASE/f"z_train_{rid}.npy"
                    zte_path = BASE/f"z_test_{rid}.npy"

                    if SKIP_IF_EXISTS and enc_path.exists() and ztr_path.exists() and zte_path.exists():
                        print(f"[SKIP] {rid} (artefactos encontrados)")
                        encoder = tf.keras.models.load_model(enc_path, compile=False)
                        z_tr_all = np.load(ztr_path); z_te_all = np.load(zte_path)
                        hist = {"loss":[np.nan], "val_loss":[np.nan]}
                        secs = np.nan
                    else:
                        print(f"[TRAIN AE] {rid}")
                        ae, hist, secs = train_autoencoder_cfg(x_tr, x_val, cfg, epochs=EPOCHS_AE, batch=BATCH_AE, verbose=0)
                        # Guardar
                        ae.encoder.save(enc_path)
                        z_tr_all = get_latents(ae.encoder, x_train, batch=8)
                        z_te_all = get_latents(ae.encoder, x_test,  batch=8)
                        np.save(ztr_path, z_tr_all); np.save(zte_path, z_te_all)
                        with open(BASE/f"history_{rid}.json", "w") as f:
                            json.dump(hist, f)

                    # Probes
                    acc_lin, f1_lin = linear_probe(z_tr_all, y_train_enc, z_te_all, y_test_enc)
                    acc_mlp, f1_mlp = mlp_probe(z_tr_all, y_train_enc, z_te_all, y_test_enc, seed=s)

                    rows.append({
                        "run_id": rid, "latent_dim": dim, "noise_sigma": nz, "l2_reg": l2, "base_ch": bc, "seed": s,
                        "ae_time_s": secs,
                        "ae_best_val_loss": (np.nanmin(hist["val_loss"]) if len(hist["val_loss"])>0 else np.nan),
                        "acc_linear": acc_lin, "f1_linear": f1_lin,
                        "acc_mlp": acc_mlp, "f1_mlp": f1_mlp
                    })
                    pd.DataFrame(rows).to_csv(BASE/"results_sweep_live.csv", index=False)

results = pd.DataFrame(rows).sort_values(["latent_dim","noise_sigma","l2_reg","seed"])
results.to_csv(BASE/"results_sweep.csv", index=False)
results

## Selección de mejor modelo de Auto Encoder

Se realiza una selección multiobjetivo usando criterio ε-Pareto:
(ε_acc = 1 punto porcentual, ε_f1 = 2 pp)

In [ ]:
# ---------------------------
# 9) Selección por accuracy (sin F1)
#    Modo A: criterio ε-accuracy
#    Modo B: criterio ratio (latent_dim / accuracy)
# ---------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

res = pd.read_csv(BASE/"results_sweep.csv")

# Elige sólo una de estas dos líneas:
SELECT_METHOD = "epsilon"   # opciones: "epsilon" o "ratio"
# SELECT_METHOD = "ratio"

KEY_ACC = "acc_mlp"         # o "acc_linear" si prefieres el linear probe
EPS_ACC = 0.01              # 1 punto porcentual (0.01 = 1%)

# --- Cálculos agregados por dim (promedio sobre runs/semillas/configs) ---
agg = (res.groupby("latent_dim")[[KEY_ACC]]
          .agg({KEY_ACC:"mean"})
          .reset_index()
          .rename(columns={KEY_ACC:"acc_mean"}))

best_acc = agg["acc_mean"].max()

if SELECT_METHOD == "epsilon":
    # MODO A: elige la menor dim con accuracy dentro de EPS_ACC del mejor
    agg["eligible"] = (agg["acc_mean"] >= best_acc - EPS_ACC)
    eligible_dims = agg[agg["eligible"]].sort_values("latent_dim")
    if not eligible_dims.empty:
        chosen_dim = int(eligible_dims.iloc[0]["latent_dim"])
    else:
        # Fallback: mejor accuracy y, si empata, menor dim
        chosen_dim = int(agg.sort_values(["acc_mean","latent_dim"], ascending=[False, True]).iloc[0]["latent_dim"])

elif SELECT_METHOD == "ratio":
    # MODO B: minimiza latent_dim / accuracy_mean
    agg["ratio_dim_acc"] = agg["latent_dim"] / np.clip(agg["acc_mean"], 1e-9, None)
    chosen_dim = int(agg.sort_values(["ratio_dim_acc","latent_dim"], ascending=[True, True]).iloc[0]["latent_dim"])

print(f"Dimensión seleccionada ({SELECT_METHOD}): {chosen_dim}")

display(agg.sort_values("latent_dim"))

# Gráficos
plt.figure(figsize=(6,4))
plt.plot(agg["latent_dim"], agg["acc_mean"], marker="o")
plt.axvline(chosen_dim, linestyle="--")
plt.title("Accuracy medio vs Dimensión latente")
plt.xlabel("Dim. latente"); plt.ylabel("Accuracy (media)")
plt.tight_layout(); plt.show()

if SELECT_METHOD == "ratio":
    plt.figure(figsize=(6,4))
    plt.plot(agg["latent_dim"], agg["ratio_dim_acc"], marker="o")
    plt.axvline(chosen_dim, linestyle="--")
    plt.title("Ratio (dim / acc_mean) vs Dimensión latente")
    plt.xlabel("Dim. latente"); plt.ylabel("dim / accuracy")
    plt.tight_layout(); plt.show()


## Entrenamiento del Clasificador

(elige el run_id de esa dim con mejor acc_mlp)

In [ ]:
# ---------------------------
# 10) Entrena/Evalúa modelo final con la dim elegida
#     (elige el run_id con MEJOR accuracy dentro de esa dim)
# ---------------------------

# Filtra sólo filas con la dimensión elegida
sub = res[res["latent_dim"] == chosen_dim].copy()

# Ordena por accuracy (usa la misma métrica que KEY_ACC)
best_row = sub.sort_values(KEY_ACC, ascending=False).iloc[0]
rid = best_row["run_id"]
print("Mejor configuración (accuracy) dentro de la dim elegida:", rid)
print(f"Accuracy de esa config: {best_row[KEY_ACC]:.4f}")

# Cargar latentes de esa run y evaluar un MLP final (misma arquitectura del probe)
z_train_ch = np.load(BASE/f"z_train_{rid}.npy")
z_test_ch  = np.load(BASE/f"z_test_{rid}.npy")

# Reusa la función mlp_probe del notebook (devuelve accuracy y entrena con early stopping)
acc_final, _ = mlp_probe(z_train_ch, y_train_enc, z_test_ch, y_test_enc, seed=SEED)
print(f"[FINAL] Accuracy TEST = {acc_final:.4f}")

# Guardar artefactos de la configuración ganadora
np.save(BASE/f"z_train_{rid}_FINAL.npy", z_train_ch)
np.save(BASE/f"z_test_{rid}_FINAL.npy",  z_test_ch)
with open(BASE/f"winner_{rid}.txt", "w") as f:
    f.write(f"winner run_id: {rid}\nacc_test: {acc_final:.4f}\n")
print("Artefactos finales guardados.")
